# Test engine nemotron tren Kaggle (language_source, VAD hysteresis)

**Truoc khi chay**

1. Session Settings: Accelerator **GPU T4 x2**, **Internet ON**.
2. Add Data: hai Kaggle Dataset cua ban:
   - `models`: noi dung thu muc `models/` (xem `scripts/copy-models.sh`): `nemotron-3.5-asr-streaming-0.6b/*.nemo`, `envit5_ctranslate2/`, `piper/`, `silero/silero_vad.jit`.
   - `source`: zip cua repo khong co `models/`, `wheels_gpu/`, `.git` (lenh zip nam trong `setup_kaggle.sh`).
3. Sua `MODELS_DATASET`, `SRC_DATASET` o cell dau tien.

Cac buoc test (bien `STEPS`): `A` auto, `B` language_source=vi, `C` language=vi, `D` neg_threshold 0.5, `E` vi/en xen ke (chi phi doi prompt), `T` dich (dinh tinh), `V` probe VAD. Cac buoc `N`, `H`, `G` (gate G0: resemble denoiser/enhancer truoc ASR) nam o cac cell cuoi notebook, can chay them `setup_enhance.sh`.

Luu y: T4 khong phai L4 nen latency/RTF khong dai dien cho production; WER va so sanh A/B/C/D thi dung duoc.

In [ ]:
import os, subprocess, textwrap

MODELS_DATASET = "models"   # ten dataset trong /kaggle/input
SRC_DATASET = "source"      # ten dataset chua source (zip hoac thu muc da giai nen)

STEPS = "A B C T V D E"     # tap con bat ky
LIMIT = 0                   # 0 = du 200 clip; dat 20 de chay thu nhanh
CONCURRENCY = 4

ENGINE_DIR = "/kaggle/working/engine"

def sh(cmd, check=True):
    print("$", cmd)
    proc = subprocess.run(["bash", "-lc", cmd], text=True)
    if check and proc.returncode != 0:
        raise RuntimeError(f"exit {proc.returncode}: {cmd}")

sh("ls /kaggle/input; nvidia-smi -L")

In [ ]:
# Tim thu muc dataset (Kaggle co the dat dataset duoi /kaggle/input/<ten> hoac /kaggle/input/datasets/<owner>/<ten>)
def find_dataset(name):
    hits = []
    for root, dirs, _ in os.walk("/kaggle/input"):
        if os.path.basename(root) == name:
            hits.append(root)
        if root.count(os.sep) >= 5:
            dirs[:] = []
    if not hits:
        raise FileNotFoundError(f"khong thay dataset '{name}' trong /kaggle/input")
    return sorted(hits, key=len)[0]

models_dir = find_dataset(MODELS_DATASET)
src_dir = find_dataset(SRC_DATASET)
print("models:", models_dir)
print("source:", src_dir)

In [ ]:
# Chep source sang noi ghi duoc (giai nen neu la zip)
sh(f"rm -rf {ENGINE_DIR} && mkdir -p {ENGINE_DIR}")
zips = [f for f in os.listdir(src_dir) if f.endswith(".zip")]
if zips:
    sh(f"cd {ENGINE_DIR} && unzip -q '{src_dir}/{zips[0]}'")
else:
    sh(f"cp -a '{src_dir}/.' {ENGINE_DIR}/")

# neu zip co thu muc goc duy nhat thi nhay vao trong
entries = os.listdir(ENGINE_DIR)
if "scripts" not in entries and len(entries) == 1:
    ENGINE_DIR = os.path.join(ENGINE_DIR, entries[0])
assert os.path.exists(f"{ENGINE_DIR}/scripts/kaggle/setup_kaggle.sh"), "khong thay scripts/kaggle trong source"
sh(f"ls {ENGINE_DIR}")

In [ ]:
# Cai dat (vai chuc phut lan dau): venv Python 3.12, torch cu124, lock, piper-tts, patch NeMo
sh(f"cd {ENGINE_DIR} && MODELS_DIR='{models_dir}' bash scripts/kaggle/setup_kaggle.sh")

In [ ]:
# Chay test. Moi lan khoi dong engine nap lai toan bo model (vai phut).
# Chay thu nhanh: LIMIT = 20, STEPS = "A B"
env = f"STEPS='{STEPS}' LIMIT={LIMIT} CONCURRENCY={CONCURRENCY}"
sh(f"cd {ENGINE_DIR} && {env} bash scripts/kaggle/run_tests_kaggle.sh")

In [ ]:
# Ket qua tong hop
print(open("/kaggle/working/out/summary.txt", encoding="utf-8").read())

In [ ]:
# Xem mot so mau hypothesis vs reference (doi ten file neu can)
py = "/kaggle/working/venv/bin/python"
sh(f"{py} {ENGINE_DIR}/scripts/kaggle/eval_wer.py show /kaggle/working/out/B_source_vi.jsonl -n 5", check=False)

In [ ]:
# Kiem tra log: moi stream mo ra voi source_mode nao (can field nay co trong log JSON)
sh("grep -h STREAM_OPEN /kaggle/working/out/logs/main/nemotron-engine.log | head -n 3", check=False)
sh("ls -la /kaggle/working/out /kaggle/working/out/logs/*", check=False)

## G0: denoiser / enhancer (resemble-enhance) truoc ASR

Do WER cua ba nhanh tren cung bo 200 clip tieng Viet, cung ASR cua engine: `A` raw (narrowband G.711), `N` denoiser ONNX, `H` enhancer (nfe 32, lambd 1.0, tau 0.5, midpoint). Them `G` do toc do GPU (denoiser, enhancer nfe 32 va 64, chunk 2 s va 10 s).

- Chay sau khi cell `setup_kaggle.sh` (cell 4) da xong. Can **Internet ON**: `setup_enhance.sh` tai checkpoint 713 MB, ONNX 42 MB va source o commit co dinh, moi file deu kiem sha256.
- Tien xu ly chay trong venv rieng (`/kaggle/working/venv_enh`) TRUOC khi engine khoi dong, nen khong dong vao venv cua engine.
- WER la can tren offline: khong tinh do tre troi cua enhancer (-11..+23 ms moi cua so 5 s) nen chua kiem duoc dieu kien dung cho luong thoi gian thuc.
- T4 khong dai dien cho GPU production: so RTF/GPU chi de so sanh tuong doi, G0 muc 2 van can do tren server that.

In [ ]:
# Cai venv enhancer + tai assets (vai phut; tai ~760 MB lan dau)
sh(f"cd {ENGINE_DIR} && bash scripts/kaggle/setup_enhance.sh")

In [ ]:
# WER 3 nhanh: A raw, N denoiser, H enhancer. Chay thu nhanh: LIMIT = 20.
# Neu A da co trong /kaggle/working/out thi doi thanh "N H" (bang so sanh van dung A_auto.jsonl cu, cung LIMIT).
ENH_STEPS = "A N H"
env = f"STEPS='{ENH_STEPS}' LIMIT={LIMIT} CONCURRENCY={CONCURRENCY}"
sh(f"cd {ENGINE_DIR} && {env} bash scripts/kaggle/run_tests_kaggle.sh")

In [ ]:
# Toc do GPU cua denoiser va enhancer (khong khoi dong engine)
sh(f"cd {ENGINE_DIR} && STEPS='G' bash scripts/kaggle/run_tests_kaggle.sh")

In [ ]:
# Ket qua G0: WER (A vs N vs H), thong ke tien xu ly (RTF, doi nang luong, % digital zero), toc do GPU
out = "/kaggle/working/out"
for name in ("summary.txt", "enhance_N_denoiser.txt", "enhance_H_enhancer.txt", "enhance_bench.txt"):
    path = f"{out}/{name}"
    print("=" * 20, name)
    print(open(path, encoding="utf-8").read() if os.path.exists(path) else "(chua co)")

In [ ]:
# Mau hypothesis cua nhanh enhancer (enhancer sinh am thanh nen co the doi tu: xem ky)
py = "/kaggle/working/venv/bin/python"
sh(f"{py} {ENGINE_DIR}/scripts/kaggle/eval_wer.py show /kaggle/working/out/H_enhancer.jsonl -n 8", check=False)